# Recompensa do GRPO

Valida a recompensa descrita em `docs/grpo.md` com exemplos: cada resposta recebe
**correctness** (−1, 0 ou 1, sem parcial), **readability** (0 a 1) e **efficiency** (0 a 1),
e a recompensa final é a soma ponderada. Readability e efficiency só contam quando a
resposta passa em todos os testes; nas outras, a legibilidade aparece só em
`legibilidade_bruta`, para inspeção.

O código das respostas roda no sandbox (`bwrap`, sem rede), como no treino.

## Configuração

In [1]:
import pandas as pd

from train.rl.reward import Weights, combine, reward_for

PESOS = Weights(correctness=1.0, readability=0.2, efficiency=0.2)
pd.set_option("display.max_colwidth", 80)


def avaliar(respostas: dict[str, str | None], exercicio: dict, recompensa=None) -> pd.DataFrame:
    """Uma linha por resposta. Use a mesma ``recompensa`` para manter as referências da efficiency."""
    recompensa = recompensa or reward_for("python")
    linhas = []
    for nome, codigo in respostas.items():
        notas = recompensa.score(codigo, exercicio)
        linhas.append({"resposta": nome, "correctness": round(notas.correctness, 3),
                       "readability": round(notas.readability, 2), "efficiency": round(notas.efficiency, 3),
                       "recompensa": round(combine(notas, PESOS), 3), **notas.details})
    return pd.DataFrame(linhas).set_index("resposta")

## Exemplos do documento

O exercício `absolute(x)`, com 5 testes. A referência da efficiency é a solução do
exercício, e as instruções contadas são só as do código da resposta.

In [2]:
ABSOLUTE = {
    "id": "absolute-exemplo",
    "name": "absolute",
    "prompt": "Write a function absolute(x) that returns the absolute value of x without using abs.",
    "tests": ["assert absolute(-3) == 3", "assert absolute(4) == 4", "assert absolute(0) == 0",
              "assert absolute(7) == 7", "assert absolute(-9) == 9"],
    "response": "def absolute(x):\n    if x < 0:\n        return -x\n    return x",
}

respostas = {
    "ideal": '''def absolute(x: int) -> int:
    """Valor absoluto."""
    # inverte o sinal dos negativos
    if x < 0:
        return -x
    return x''',
    "sem tipos": "def absolute(x):\n    if x < 0:\n        return -x\n    return x",
    "laço desnecessário": "def absolute(x):\n    result = x\n    for _ in range(50):\n        pass\n    if x < 0:\n        result = -x\n    return result",
    "return x": "def absolute(x):\n    return x",
    "usa abs (proibido)": "def absolute(x):\n    return abs(x)",
    "variável não usada": "def absolute(x):\n    lixo = 1\n    if x < 0:\n        return -x\n    return x",
    "sintaxe inválida": "def absolute(x:\n    return",
    "erro ao carregar": "def absolute(x):\n    return x\nfoo()",
    "laço infinito": "def absolute(x):\n    while True:\n        pass",
    "vazia": None,
}
avaliar(respostas, ABSOLUTE)

,correctness,readability,efficiency,recompensa,testes,instrucoes,legibilidade_bruta,tipos,docstring,linter,comentarios,proporcao_comentarios,referencia,motivo
resposta,,,,,,,,,,,,,,
ideal,1.0,1.00,1.000,1.400,5/5,32.0,1.000000,1.0,1.0,1.000000,1.0,0.166667,32.0,NaN
sem tipos,1.0,0.25,1.000,1.250,5/5,32.0,0.250000,0.0,0.0,1.000000,0.0,0.000000,32.0,NaN
laço desnecessário,1.0,0.25,0.039,1.058,5/5,821.0,0.250000,0.0,0.0,1.000000,0.0,0.000000,32.0,NaN
return x,0.0,0.00,0.000,0.000,3/5,10.0,0.250000,0.0,0.0,1.000000,0.0,0.000000,NaN,NaN
usa abs (proibido),0.0,0.00,0.000,0.000,5/5,20.0,0.250000,0.0,0.0,1.000000,0.0,0.000000,NaN,usa nome proibido: abs
variável não usada,1.0,0.17,0.762,1.186,5/5,42.0,0.166667,0.0,0.0,0.666667,0.0,0.000000,32.0,NaN
sintaxe inválida,-1.0,0.00,0.000,-1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,incompleto ou sintaxe inválida
erro ao carregar,-1.0,0.00,0.000,-1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,erro ao carregar: NameError
laço infinito,-1.0,0.00,0.000,-1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tempo esgotado


## Teste o seu exemplo

Edite o exercício e as respostas abaixo. `name` é a função pedida (`None` se a resposta
é em nível de módulo), e `response` é a solução usada como referência da efficiency.

In [3]:
MEU_EXERCICIO = {
    "id": "meu-exemplo",
    "name": "soma_lista",
    "prompt": "Escreva uma função soma_lista(itens) que retorna a soma dos números da lista sem usar sum.",
    "tests": ["assert soma_lista([]) == 0", "assert soma_lista([1, 2, 3]) == 6", "assert soma_lista([-1, 1]) == 0"],
    "response": "def soma_lista(itens):\n    total = 0\n    for item in itens:\n        total += item\n    return total",
}
minhas_respostas = {
    "laço": "def soma_lista(itens):\n    total = 0\n    for item in itens:\n        total += item\n    return total",
    "usa sum": "def soma_lista(itens):\n    return sum(itens)",
    "recursiva": "def soma_lista(itens):\n    if not itens:\n        return 0\n    return itens[0] + soma_lista(itens[1:])",
    "errada": "def soma_lista(itens):\n    return len(itens)",
}
avaliar(minhas_respostas, MEU_EXERCICIO)

,correctness,readability,efficiency,recompensa,testes,instrucoes,legibilidade_bruta,tipos,docstring,linter,comentarios,proporcao_comentarios,referencia,motivo
resposta,,,,,,,,,,,,,,
laço,1.0,0.25,1.000,1.250,3/3,56,0.25,0.0,0.0,1.0,0.0,0.0,56.0,NaN
usa sum,0.0,0.00,0.000,0.000,3/3,12,0.25,0.0,0.0,1.0,0.0,0.0,NaN,usa nome proibido: sum
recursiva,1.0,0.25,0.757,1.201,3/3,74,0.25,0.0,0.0,1.0,0.0,0.0,56.0,NaN
errada,0.0,0.00,0.000,0.000,1/3,12,0.25,0.0,0.0,1.0,0.0,0.0,NaN,NaN


## Exercícios da validação

Solução de referência contra respostas inúteis, em alguns exercícios reais (modelos
nunca vistos no treino). A solução deve receber correctness 1, e as inúteis ficam
perto de 0.

In [4]:
from eval.exercises import load_exercises

exercicios = load_exercises(per_concept=1)[:6]
tabelas = []
for exercicio in exercicios:
    inuteis = {"solução": exercicio["response"]}
    if exercicio["name"]:
        inuteis["return None"] = f"def {exercicio['name']}(*args):\n    return None"
        inuteis["primeiro argumento"] = f"def {exercicio['name']}(*args):\n    return args[0]"
    tabela = avaliar(inuteis, exercicio)
    tabela.insert(0, "exercício", exercicio["template"])
    tabelas.append(tabela)
pd.concat(tabelas).reindex(columns=["exercício", "correctness", "readability", "efficiency", "recompensa", "testes", "motivo"])

,exercício,correctness,readability,efficiency,recompensa,testes,motivo
resposta,,,,,,,
solução,opencoder-next_largest_primes,1.0,0.25,1.0,1.25,3/3,NaN
return None,opencoder-next_largest_primes,0.0,0.00,0.0,0.00,0/3,NaN
primeiro argumento,opencoder-next_largest_primes,0.0,0.00,0.0,0.00,0/3,NaN
solução,positives,1.0,0.25,1.0,1.25,5/5,NaN
return None,positives,0.0,0.00,0.0,0.00,0/5,NaN
primeiro argumento,positives,0.0,0.00,0.0,0.00,2/5,NaN
solução,char_count,1.0,0.25,1.0,1.25,5/5,NaN
return None,char_count,0.0,0.00,0.0,0.00,0/5,NaN
primeiro argumento,char_count,0.0,0.00,0.0,0.00,0/5,NaN


## Respostas reais do modelo no MBPP

Respostas do `base-ex-v3` já gravadas pelo benchmark (`eval.benchmark run`), avaliadas
com os testes oficiais. Troque `CHECKPOINT` para comparar modelos.

In [5]:
import json
import re

from datasets import load_dataset

from eval.benchmark import variant_path

CHECKPOINT = "base-ex-v3/step-00762"
QUANTOS = 8

oficiais = {f"mbpp-{r['task_id']}": r for r in load_dataset("google-research-datasets/mbpp", "full", split="test")}
gravadas = [json.loads(l) for l in variant_path(CHECKPOINT, "gguf", True).read_text().splitlines()]
gravadas = [r for r in gravadas if r["suite"] == "mbpp"]
escolhidas = [r for r in gravadas if r["passed"]][: QUANTOS // 2] + [r for r in gravadas if not r["passed"]][: QUANTOS // 2]

recompensa = reward_for("python")
tabelas = []
for gravada in escolhidas:
    oficial = oficiais[gravada["id"]]
    funcoes = re.findall(r"def\s+(\w+)", oficial["code"])
    exercicio = {"id": gravada["id"], "prompt": oficial["text"], "response": oficial["code"],
                 "tests": [oficial["test_setup_code"] + "\n" + t if oficial["test_setup_code"] else t for t in oficial["test_list"]],
                 "name": next((f for f in funcoes if f in oficial["test_list"][0]), None)}
    tabela = avaliar({gravada["id"]: gravada["code"]}, exercicio, recompensa)
    tabela.insert(0, "enunciado", oficial["text"][:70])
    tabelas.append(tabela)
pd.concat(tabelas).reindex(columns=["enunciado", "correctness", "readability", "efficiency", "recompensa", "testes", "instrucoes", "referencia", "motivo"])

,enunciado,correctness,readability,efficiency,recompensa,testes,instrucoes,referencia,motivo
resposta,,,,,,,,,
mbpp-12,Write a function to sort a given matrix in ascending order according t,1.0,0.25,0.421,1.134,3/3,57,24.0,NaN
mbpp-19,Write a function to find whether a given array of integers contains an,1.0,0.25,0.235,1.097,3/3,153,36.0,NaN
mbpp-22,Write a function to find the first duplicate element in a given array,1.0,0.25,1.000,1.250,3/3,145,208.0,NaN
mbpp-25,Write a python function to find the product of non-repeated elements i,1.0,0.25,1.000,1.250,3/3,129,287.0,NaN
mbpp-11,Write a python function to remove first and last occurrence of a given,0.0,0.00,0.000,0.000,0/3,15,NaN,NaN
mbpp-13,Write a function to count the most common words in a dictionary.,0.0,0.00,0.000,0.000,0/3,227,NaN,NaN
mbpp-14,Write a python function to find the volume of a triangular prism.,0.0,0.00,0.000,0.000,0/3,30,NaN,NaN
mbpp-15,Write a function to split a string at lowercase letters.,0.0,0.00,0.000,0.000,2/3,253,NaN,NaN


In [6]:
# Uma resposta por inteiro: troque o índice
print(escolhidas[0]["code"])

def sort_matrix(matrix):
    return sorted(matrix, key=lambda x: sum(x))

